In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path(r"C:\Users\Gebruiker\OneDrive\Spain\Paper 1\precipitation_assessment\DATA_DIR")

# Load and filter to study period 2001-2020
df = pd.read_csv(DATA_DIR / "merged_obs_grid.csv")
print(f"Before: {df.shape}")

df = df[(df['year'] >= 2001) & (df['year'] <= 2020)]
print(f"After year filter: {df.shape}")  # should be 3600

# Fix MERRA2 — rebuild from yearly files
yearly = sorted(DATA_DIR.glob("precip_extraction_MERRA2_????.csv"))
print(f"\nFound {len(yearly)} MERRA2 yearly files")

dfs = []
for f in yearly:
    yr = int(f.stem.split("_")[-1])
    if yr < 2001 or yr > 2020:
        continue
    raw = pd.read_csv(f)
    # Find the precip column (not metadata)
    skip = {"system:index",".geo","station_id","product","year","month"}
    pcol = [c for c in raw.columns if c not in skip and raw[c].dtype != object]
    if not pcol:
        print(f"  ⚠ No numeric column in {f.name}")
        continue
    pcol = pcol[0]
    print(f"  {f.name}  precip_col='{pcol}'")
    raw = raw.rename(columns={pcol: "precip_mm_day"})
    raw = raw.dropna(subset=["precip_mm_day"])
    raw = raw[raw["year"].astype(int) == yr]
    raw["year"]  = raw["year"].astype(int)
    raw["month"] = raw["month"].astype(int)
    dfs.append(raw[["station_id","year","month","precip_mm_day"]])

merra = pd.concat(dfs, ignore_index=True).drop_duplicates(
    subset=["station_id","year","month"]
)
print(f"\nMERRA2 rebuilt: {len(merra)} rows (expected 3600)")
print(f"NaN: {merra['precip_mm_day'].isna().sum()}")

# Drop old MERRA2 column and merge new one
df = df.drop(columns=["MERRA2"])
df = df.merge(
    merra.rename(columns={"precip_mm_day":"MERRA2"}),
    on=["station_id","year","month"],
    how="left"
)

# Final check
print(f"\nFinal shape: {df.shape}")
print("NaN per column:")
for col in df.columns:
    n = df[col].isna().sum()
    print(f"  {'⚠' if n>0 else '✓'}  {col:<18} NaN={n}")

# Save
df.to_csv(DATA_DIR / "merged_obs_grid.csv", index=False)
print("\n✅ Saved merged_obs_grid.csv")
print("NEXT: python validation_metrics.py")

Shape: (3780, 10)
Columns: ['station_id', 'year', 'month', 'obs_mm_day', 'CHIRPS', 'ERA5_LAND', 'GPM_IMERG', 'MERRA2', 'PERSIANN_CDR', 'TERRACLIMATE']

Expected shape: (3600, 10)
  3600 = 15 stations × 20 years × 12 months
  10 cols = station_id, year, month, obs_mm_day, 6 products

--- Year range ---
  Min: 2000  Max: 2020

--- Stations ---
['WA001', 'WA002', 'WA003', 'WA004', 'WA005', 'WA006', 'WA007', 'WA008', 'WA009', 'WA010', 'WA011', 'WA012', 'WA013', 'WA014', 'WA015']

--- NaN per column ---
  ✓  station_id          NaN=   0  (0.0%)
  ✓  year                NaN=   0  (0.0%)
  ✓  month               NaN=   0  (0.0%)
  ✓  obs_mm_day          NaN=   0  (0.0%)
  ✓  CHIRPS              NaN=   0  (0.0%)
  ✓  ERA5_LAND           NaN=   0  (0.0%)
  ✓  GPM_IMERG           NaN=  75  (2.0%)
  ⚠  MERRA2              NaN=3600  (95.2%)
  ✓  PERSIANN_CDR        NaN=   0  (0.0%)
  ✓  TERRACLIMATE        NaN=   0  (0.0%)

--- Mean mm/day per product ---
  CHIRPS              mean=3.804  std=5.07